# LQ control checked by a derivation ledger

This notebook derives the optimal control of a one-period
linear-quadratic (LQ) problem with
[sympy_matrix_tools](https://github.com/JRF-2018/sympy_matrix_tools)
and checks every step with a `Derivation` ledger.

The state $y$ is given by $y = A y_0 + B x + z$, where $x$ is the
control. We minimize
$J = \frac{1}{2} y^T W y + p^T y$ with respect to $x$, where $W$ is
symmetric. The answer is $x = -K^{-1} B^T (W (A y_0 + z) + p)$ with
$K = B^T W B$.

This is a simplified version of a derivation in
[economy_control](https://github.com/JRF-2018/economy_control).

In [ ]:
!pip install "sympy>=1.14" git+https://github.com/JRF-2018/sympy_matrix_tools

In [2]:
from sympy import *
from sympy_matrix_tools import *
from sympy_matrix_tools.derivation import Derivation, Grad, cas, rule
init_printing()

## Symbols

The dimensions `n` and `m` are symbolic. `Q` is SymPy's namespace of
assumptions, so the weight matrix is called `W`.

In [3]:
n, m = symbols("n m", integer=True, positive=True)
A = MatrixSymbol("A", n, n)
B = MatrixSymbol("B", n, m)
W = MatrixSymbol("W", n, n)
y0 = MatrixSymbol("y0", n, 1)
z = MatrixSymbol("z", n, 1)
p = MatrixSymbol("p", n, 1)
x = MatrixSymbol("x", m, 1)
K = MatrixSymbol("K", m, m)

## Hypotheses and definitions

A hypothesis such as the symmetry of `W` or the invertibility of `K`
is respected when random values are generated for the numerical
checks. A definition makes `K` an abbreviation.

In [4]:
d = Derivation("One-period LQ control")
d.hypothesis(Eq(W.T, W), "W_sym")
d.hypothesis(Q.invertible(K), "K_inv")
d.define(K, B.T*W*B, "K_def")

## A derived fact: K is symmetric

Each step is written explicitly, as in the `calc` of Lean, and has a
justification: `rule(name)` rewrites by a named fact, and
`cas(...)` is a computation by SymPy.

In [5]:
d.calc("K_sym", K.T,
       (B.T*W.T*B, rule("K_def")),
       (B.T*W*B, rule("W_sym")),
       (K, rule("K_def")))

## The gradient

`Grad(J, x)` is the unevaluated gradient. The objective must be
written explicitly in `x`. `alias` abbreviates `J` in the reports.

In [6]:
y = A*y0 + B*x + z
J = y.T*W*y/2 + p.T*y
d.alias(J, "J")
d.calc("grad", Grad(J, x),
       (B.T*W*y/2 + B.T*W.T*y/2 + B.T*p, cas("doit")),
       (B.T*W*y + B.T*p, rule("W_sym")),
       (K*x + B.T*(W*(A*y0 + z) + p), rule("K_def")))

## The first-order condition and the solution

The first-order condition is a constraint that random values do not
satisfy, so the step using it is not checked numerically ("n/a").
`solve_linear` solves $K x + c = 0$; the invertibility of `K` is an
obligation, discharged by the hypothesis `K_inv`.

In [7]:
d.hypothesis(Eq(Grad(J, x), ZeroMatrix(m, 1), evaluate=False), "foc")
d.calc("foc_linear", K*x + B.T*(W*(A*y0 + z) + p),
       (Grad(J, x), rule("grad")),
       (ZeroMatrix(m, 1), rule("foc")))
d.solve_linear("x_opt", "foc_linear", x)

## The optimal value

With the abbreviations $v = A y_0 + z$ (the state without control)
and $c = W v + p$, the objective is a quadratic function of $x$:
$J = J_0 + x^T B^T c + \frac{1}{2} x^T K x$, where
$J_0 = \frac{1}{2} v^T W v + p^T v$. Substituting the optimal control
gives $J = J_0 - \frac{1}{2} c^T B K^{-1} B^T c$.

The symbolic check of the second step uses the fact that a $1 \times 1$
matrix equals its transpose, together with the symmetry `W_sym` cited
in the step.

In [8]:
v = MatrixSymbol("v", n, 1)
c = MatrixSymbol("c", n, 1)
d.define(v, A*y0 + z, "v_def")
d.define(c, W*v + p, "c_def")
J0 = v.T*W*v/2 + p.T*v
d.calc("J_quad", J,
       ((v + B*x).T*W*(v + B*x)/2 + p.T*(v + B*x), rule("v_def")),
       (J0 + x.T*B.T*(W*v + p) + x.T*K*x/2, rule("W_sym"), rule("K_def")),
       (J0 + x.T*B.T*c + x.T*K*x/2, rule("c_def")))

In [9]:
x_opt = -K.inverse()*B.T*c
d.alias(x_opt, "x_opt")
Jq = J0 + x.T*B.T*c + x.T*K*x/2
d.calc("J_opt", Jq.subs(x, x_opt),
       (J0 - c.T*B*K.T.inverse()*B.T*c/2, cas("doit")),
       (J0 - c.T*B*K.inverse()*B.T*c/2, rule("K_sym")))

## Checking

`check()` replaces the symbols by random rational matrices that
satisfy the hypotheses and compares both sides of every step
exactly.

In [10]:
d.check(trials=3)

True

In [11]:
print(d.report())

Derivation: One-period LQ control
[hypothesis] W_sym: W.T = W
[hypothesis] K_inv: Q.invertible(K)
[definition] K_def: K = B.T*W*B
[derived] K_sym: K.T = K
  K.T
  = B.T*W.T*B    by rule K_def    symbolic: ok    numeric: ok 3/3
  = B.T*W*B    by rule W_sym    symbolic: ok    numeric: ok 3/3
  = K    by rule K_def    symbolic: ok    numeric: ok 3/3
[derived] grad: Grad(J, x) = K*x + B.T*(W*(A*y0 + z) + p)
  Grad(J, x)
  = (1/2)*B.T*W*(A*y0 + B*x + z) + B.T*p + (1/2)*B.T*W.T*(A*y0 + B*x + z)    by cas doit    symbolic: ok    numeric: ok 3/3
  = B.T*W*(A*y0 + B*x + z) + B.T*p    by rule W_sym    symbolic: ok    numeric: ok 3/3
  = K*x + B.T*(W*(A*y0 + z) + p)    by rule K_def    symbolic: ok    numeric: ok 3/3
[hypothesis] foc: Grad(J, x) = 0
[derived] foc_linear: K*x + B.T*(W*(A*y0 + z) + p) = 0
  K*x + B.T*(W*(A*y0 + z) + p)
  = Grad(J, x)    by rule grad    symbolic: ok    numeric: ok 3/3
  = 0    by rule foc    symbolic: ok    numeric: n/a
[derived] x_opt: x = -K**(-1)*B.T*(W*(A*y0 + z

In a notebook, the ledger itself is displayed as a table.

In [12]:
d

## A wrong step is found

Suppose someone (a human or an AI) claims that SymPy computed a
gradient without the factor 1/2 in the right place. The numerical
check finds it, and a counterexample is kept.

In [13]:
d2 = Derivation("A wrong step")
d2.alias(J, "J")
d2.calc("grad", Grad(J, x),
        (B.T*W.T*y + B.T*p/2, cas("doit")))
d2.check(trials=3)

False

In [14]:
print(d2.report())

Derivation: A wrong step
[derived] grad: Grad(J, x) = (1/2)*B.T*p + B.T*W.T*(A*y0 + B*x + z)
  Grad(J, x)
  = (1/2)*B.T*p + B.T*W.T*(A*y0 + B*x + z)    by cas doit    symbolic: FAILED    numeric: FAILED 0/3
Summary: 1 step, 0 asserted, 1 symbolic failure, 1 numeric failure, 0 open obligations.
